# Phase 1: Data Exploration

This notebook explores the HMP and AGP microbiome datasets:
- Load OTU tables and metadata
- Compute basic diversity metrics
- Examine phylum-level composition
- Preview co-occurrence structure
- Generate first persistence diagrams

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from biom import load_table
from skbio.diversity import alpha_diversity, beta_diversity

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Greens_d')

## 1. Load HMP Data

In [ ]:
# Load HMP OTU table
hmp_table = load_table('../data/raw/hmp/hmp1_otu_table.biom')
print(f'HMP OTU table: {hmp_table.shape[0]} OTUs x {hmp_table.shape[1]} samples')

# Convert to pandas DataFrame (samples x OTUs)
hmp_df = pd.DataFrame(hmp_table.to_dataframe(dense=True).T)
print(f'Shape: {hmp_df.shape}')
print(f'Sparsity: {(hmp_df == 0).sum().sum() / hmp_df.size:.1%}')

## 2. Alpha Diversity Metrics

In [ ]:
# Shannon diversity per sample
shannon = alpha_diversity('shannon', hmp_df.values, hmp_df.index)
print(f'Shannon diversity: mean={shannon.mean():.2f}, std={shannon.std():.2f}')

# Simpson diversity
simpson = alpha_diversity('simpson', hmp_df.values, hmp_df.index)
print(f'Simpson diversity: mean={simpson.mean():.2f}, std={simpson.std():.2f}')

# Observed OTUs
observed = alpha_diversity('observed_otus', hmp_df.values, hmp_df.index)
print(f'Observed OTUs: mean={observed.mean():.0f}, std={observed.std():.0f}')

In [ ]:
# Plot diversity distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].hist(shannon, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Shannon Diversity')
axes[0].set_xlabel('Shannon Index')

axes[1].hist(simpson, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Simpson Diversity')
axes[1].set_xlabel('Simpson Index')

axes[2].hist(observed, bins=30, edgecolor='black', alpha=0.7)
axes[2].set_title('Observed OTUs')
axes[2].set_xlabel('Number of OTUs')

plt.tight_layout()
plt.savefig('../figures/diversity_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Phylum-level Composition

Aggregate OTU abundances to phylum level to examine the
Firmicutes/Bacteroidetes ratio and overall community structure.

In [ ]:
# Extract taxonomy metadata from BIOM table (if available)
# taxonomy = hmp_table.metadata_to_dataframe('observation')
# Aggregate to phylum level and compute relative abundances
# This will be populated once data is downloaded
print('Phylum-level aggregation: requires taxonomy metadata from BIOM table')

## 4. Load AGP Data + Metadata

In [ ]:
# Load AGP data
# agp_table = load_table('../data/raw/agp/agp_otu_table.biom')
# agp_meta = pd.read_csv('../data/raw/agp/agp_metadata.tsv', sep='\t')

# Key metadata columns for our research:
# - diet_type (connects to F/B ratio)
# - antibiotic_history (connects to dysbiosis)
# - ibd, ibs (disease states)
# - mental_health_* (connects to gut-brain axis)
# - bmi_cat (connects to F/B ratio shifts)
print('AGP data loading: requires downloaded data')

## 5. Co-occurrence Preview

Quick Spearman correlation on top 50 most abundant OTUs
as a preview of the full SparCC network.

In [ ]:
from scipy.stats import spearmanr

# Select top 50 most abundant OTUs
top_50 = hmp_df.sum().nlargest(50).index
corr_matrix, p_matrix = spearmanr(hmp_df[top_50])
corr_df = pd.DataFrame(corr_matrix, index=top_50, columns=top_50)

# Visualize
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_df, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Spearman Correlation: Top 50 OTUs')
plt.savefig('../figures/correlation_heatmap_top50.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. First Persistence Diagram

Apply Vietoris-Rips filtration on the correlation distance matrix
to get our first look at the topological structure.

In [ ]:
from ripser import ripser
from persim import plot_diagrams

# Convert correlation to distance
distance_matrix = 1 - np.abs(corr_df.values)
np.fill_diagonal(distance_matrix, 0)

# Compute persistent homology
result = ripser(distance_matrix, maxdim=2, distance_matrix=True)

print(f'H0 features: {len(result["dgms"][0])}')
print(f'H1 features: {len(result["dgms"][1])}')
if len(result['dgms']) > 2:
    print(f'H2 features: {len(result["dgms"][2])}')

# Plot persistence diagram
fig, ax = plt.subplots(figsize=(8, 6))
plot_diagrams(result['dgms'], ax=ax, show=False)
ax.set_title('Persistence Diagram: Top 50 OTU Co-occurrence Network')
plt.savefig('../figures/persistence_diagram_initial.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute Betti curves
import sys
sys.path.insert(0, '..')
from src.tda.features import betti_curve

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2196F3', '#FF5722', '#4CAF50']
labels = [r'$\beta_0$', r'$\beta_1$', r'$\beta_2$']

for dim, dgm in enumerate(result['dgms']):
    if dim >= 3:
        break
    filt_vals, betti = betti_curve(dgm, num_points=100)
    ax.plot(filt_vals, betti, color=colors[dim], label=labels[dim], linewidth=2)

ax.set_xlabel('Filtration Value')
ax.set_ylabel('Betti Number')
ax.set_title('Betti Curves: Top 50 OTU Co-occurrence Network')
ax.legend()
plt.savefig('../figures/betti_curves_initial.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Phase 1 exploration complete. Key observations:
- HMP OTU table loaded and basic statistics computed
- Diversity metrics calculated (Shannon, Simpson, observed OTUs)
- Initial co-occurrence correlation matrix computed for top 50 OTUs
- First persistence diagram generated, showing topological features

Next steps (Phase 2):
- Implement SparCC for compositional-aware correlation
- Build full genus-level co-occurrence networks
- Compare network structure across metadata groups